# Pretrain on continuous spatial cue, then stage 2

Replaces stage 0 + stage 1 of `01c_train_topo.ipynb` with a single
supervised pretraining pass on `SpatialPretrain` (cue at random uniform
(x, y) in [-1, 1]^2, on for the whole trial). The pretrained recurrent
weights are loaded into a fresh model that's then trained on the full
distractor task (stage 2 from the original topo notebook).

Adaptation is disabled (`use_adaptation=False`).


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetWithDistractorsV3, SpatialPretrain
from src.training import TrainConfig, train_supervised
from src.train_pretrain import PretrainConfig, train_spatial_pretrain

device = "cpu"
print("device:", device)
Path("checkpoints").mkdir(exist_ok=True)


In [ ]:
def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=False,  # adaptation disabled
    ).to(device)


## Pretrain on SpatialPretrain

MSE on (x, y) decoded from hidden via `model.decode_xy`. First 10 timesteps (200ms, ~89% settled given alpha=0.2) are excluded from the loss.

In [ ]:
def make_env_pretrain():
    return SpatialPretrain(dt=20, trial_len_ms=1500, cue_range=(-1.0, 1.0))


model = make_model()
cfg_pre = PretrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=5000,
    print_every=50,
    grad_clip=10.0,
    settle_steps=10,
    device=device,
)
history_pre = train_spatial_pretrain(model, make_env_pretrain, cfg_pre)
torch.save({"state_dict": model.state_dict()}, "checkpoints/pretrain_topo.pt")
print("Saved: checkpoints/pretrain_topo.pt")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_pre["update"], history_pre["loss"])
plt.xlabel("update")
plt.ylabel("MSE")
plt.title("Spatial pretraining loss")
plt.grid(alpha=0.3)
plt.show()


## Stage 2 — cue + distractors

Copied from `01c_train_topo.ipynb` cell 10, but loads `pretrain_topo.pt` instead of `stage1_topo.pt` and saves to `stage2_from_pretrain_topo.pt`. `fa_penalty_weight=0.0` matches the original.

In [ ]:
import shutil
from pathlib import Path

# One-shot backup: preserve the curriculum-trained stage2_topo.pt before
# overwriting it with the pretrain-init version. Skip if backup already exists.
_orig = Path("checkpoints/stage2_topo.pt")
_backup = Path("checkpoints/stage2_topo_original.pt")
if _orig.exists() and not _backup.exists():
    shutil.copy(_orig, _backup)
    print(f"Backed up {_orig.name} -> {_backup.name}")


def make_env_stage2():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
    )


model = make_model()
model.load_state_dict(
    torch.load("checkpoints/pretrain_topo.pt", weights_only=True)["state_dict"],
    strict=False,
)
cfg2 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=500,
    print_every=50,
    device=device,
    com_weight=0.5,
    sparsity_weight=0.01,
    fa_penalty_weight=0.0,
)
history2 = train_supervised(model, make_env_stage2, cfg2)
# Save to the canonical analysis path so the 8 topo analysis notebooks
# pick up these weights automatically. The original is preserved as
# stage2_topo_original.pt (backup above).
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage2_topo.pt")
print("Saved: checkpoints/stage2_topo.pt (analysis notebooks will use this)")


## Stage 2 training curves

In [ ]:
def smooth(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")


fig, ax = plt.subplots(1, 1, figsize=(8, 4))
updates = np.arange(1, len(history2["p_correct"]) + 1)
for key, color, label in [
    ("p_correct", "green", "correct"),
    ("p_miss", "orange", "miss"),
    ("p_abort", "red", "abort"),
]:
    vals = history2[key]
    ax.plot(updates, vals, alpha=0.15, color=color)
    if len(vals) >= 20:
        ax.plot(updates[19:], smooth(vals, 20), color=color, label=label)
ax.set_xlabel("print step")
ax.set_ylabel("fraction")
ax.set_title("Stage 2 (pretrained init)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Quick eval

In [ ]:
from src.analysis import collect_trials

trials = collect_trials(model, make_env_stage2, n_trials=2000, device=device)
outcomes = {}
for tr in trials:
    o = tr["train_outcome"]
    outcomes[o] = outcomes.get(o, 0) + 1
total = len(trials)
print(f"Total trials: {total}")
for o, n in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")
